# 🌿 BioEcoInt — Pipeline de Treinamento e Benchmark SOTA no Google Colab
> **Laboratório IFMaker / BioEcoInt** — *Engenharia de Software & Machine Learning Aplicada à Fauna Amazônica*
> 
> Este notebook executa os **3 Experimentos de Classificação da Fauna** projetados para superar o platô de acurácia da ResNet18:
> 1. **Experimento 1 (Baseline Sanado)**: ResNet18 com divisão estrita de vídeos (zero leakage), LR diferencial por camadas e Two-Stage Warmup.
> 2. **Experimento 2 (SOTA 1 - Two-Stage YOLO Crop)**: YOLOv8n ultraleve para detecção da Bounding Box do animal + Recorte com margem dinâmica (+15% padding) + ResNet18 focada exclusivamente na ROI.
> 3. **Experimento 3 (SOTA 2 - ResNet18 Coordinate Attention)**: ResNet18 re-arquitetada com mecanismos de Atenção Coordenada 2D (Hou et al., CVPR) nas camadas residuais profundas (resguardando alvos pequenos e camuflados).
> 
> ---
> ⚡ **Gerenciamento de Compute Units (CUs)**: Este pipeline utiliza salvamento contínuo de checkpoints e encerramento automatizado de runtime (`runtime.unassign()`) para evitar desperdício de créditos do seu plano Google AI Pro (200 CUs).


### 1. Verificação de Aceleração por Hardware (GPU)
Certifique-se de que o ambiente de execução está configurado com GPU (T4, L4, V100 ou A100).


In [ ]:
import torch
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo: {torch.cuda.get_device_name(0)}")
    print(f"Capacidade de Memória: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
!nvidia-smi


### 2. Instalação do Gerenciador de Pacotes Ultrarrápido (`uv`)
Utilizamos o `uv` da Astral para provisionar o ambiente virtual e dependências em menos de 40 segundos, garantindo isolamento total.


In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"
!uv --version


### 3. Clonagem do Repositório (`bio-interaction-ml`)
Configure seu Personal Access Token do GitHub nos Secrets do Colab (`GITHUB_TOKEN`) ou utilize acesso direto caso o repositório seja público.


In [ ]:
import os
from google.colab import userdata

try:
    github_token = userdata.get('GITHUB_TOKEN')
    repo_url = f"https://{github_token}@github.com/Emmerson2018/bio-interaction-ml.git"
    print("✓ Autenticação via GitHub Token configurada com sucesso.")
except Exception:
    repo_url = "https://github.com/Emmerson2018/bio-interaction-ml.git"
    print("ℹ GITHUB_TOKEN não encontrado nos Secrets do Colab. Tentando clonagem pública...")

if not os.path.exists("bio-interaction-ml"):
    !git clone {repo_url} bio-interaction-ml
else:
    print("Repositório já clonado. Atualizando...")
    !cd bio-interaction-ml && git pull

%cd bio-interaction-ml
!git status


### 4. Sincronização do Ambiente Virtual (`uv sync`)
Instala PyTorch com aceleração CUDA, Torchvision, Ultralytics, Optuna, ONNX, DVC e demais dependências da toolbox.


In [ ]:
!uv sync
!uv run python -c "import torch, torchvision, ultralytics; print('✓ Ambiente provisionado com sucesso! PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())" 


### 5. Montagem do Google Drive & Sincronização do Dataset Sanado via DVC
Montamos o Google Drive para obter as credenciais e sincronizar o dataset sanitizado (80 vídeos, 15 classes, 19.413 imagens geradas com zero vazamento entre vídeos).
O dataset é descompactado diretamente no SSD NVMe local (`/content/bio-interaction-ml/datasets/generated/`) para máxima taxa de leitura por segundo (eliminando gargalos de I/O do FUSE).


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Configurar segredo local do OAuth Client do laboratório no DVC
!uv run dvc remote modify --local gdrive-storage gdrive_client_secret "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"

# Puxar o arquivo compactado do dataset sanitizado
!uv run dvc pull datasets/generated/animals_video_sanitized.zip.dvc

# Descompactar no SSD efêmero local para leitura ultrarrápida durante o treino
if os.path.exists("datasets/generated/animals_video_sanitized.zip"):
    print("Descompactando dataset sanitizado no disco local...")
    !unzip -q -o datasets/generated/animals_video_sanitized.zip
    # Salvaguarda para desaninhamento se já tiver sido descompactado anteriormente
    if os.path.exists("datasets/generated/datasets/generated/animals_video_sanitized"):
        !mv datasets/generated/datasets/generated/animals_video_sanitized datasets/generated/
    !ls -lh datasets/generated/animals_video_sanitized
    print("✓ Dataset pronto para treinamento!")
else:
    print("⚠️ Arquivo zip não encontrado localmente via DVC. Verifique as credenciais do Google Drive.")


### 6. 🚀 Experimento 1: ResNet18 Baseline Sanado
- **Arquitetura**: ResNet18 pré-treinada com Two-Stage Transfer Learning.
- **Divisão**: Vídeos estritamente disjuntos por classe (Zero Shuffle Leakage).
- **Otimizador**: AdamW com taxas de aprendizado diferenciais ($1.5 \times 10^{-5}$ nas camadas rasas, $4.5 \times 10^{-5}$ nas profundas e $4 \times 10^{-4}$ no classificador).
- **Augmentation**: Mixup & CutMix calibrados.


In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp1_resnet18_baseline.yml


### 7. 🎯 Experimento 2 (SOTA 1): Two-Stage YOLO Crop Classifier
- **Arquitetura**: Detector YOLOv8n (estágio 1) localiza a ROI da fauna com margem dinâmica (+15% padding) $\to$ ResNet18 (estágio 2) classifica exclusivamente o animal.
- **Benefício**: Elimina ruído de fundo amazônico denso, troncos e folhagens que confundem alvos camuflados.


In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp2_resnet18_yolo_crop.yml


### 8. 🧠 Experimento 3 (SOTA 2): ResNet18 com Coordinate Attention (CA)
- **Arquitetura**: ResNet18 re-arquitetada integrando blocos de Atenção Coordenada 2D (Hou et al., CVPR) após as convoluções dos estágios Layer3 e Layer4.
- **Benefício**: Codifica simultaneamente relações espaciais horizontais e verticais de longo alcance sem o custo quadrático de Vision Transformers, preservando pequenos animais em resolução mobile.


In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp3_resnet18_ca.yml


### 9. 📱 Exportação Mobile e Quantização (TFLite FP16 & INT8 PTQ)
Converte os melhores modelos treinados para formatos otimizados de borda (ONNX e TFLite), medindo tamanho e latência de inferência sob as restrições mobile do projeto.


In [ ]:
# Exportando os modelos dos 3 experimentos para TFLite FP16 e INT8
!MPLBACKEND=Agg uv run base-export -opt options/exp1_resnet18_baseline.yml
!MPLBACKEND=Agg uv run base-export -opt options/exp2_resnet18_yolo_crop.yml
!MPLBACKEND=Agg uv run base-export -opt options/exp3_resnet18_ca.yml


### 10. 📊 Comparativo Consolidado de Métricas dos 3 Experimentos
Gera tabela e gráficos comparando acurácia balanceada, Macro F1, tamanho em disco e taxa de acerto por classe.


In [ ]:
import os
import glob
import json
import pandas as pd
import matplotlib.pyplot as plt

exp_dirs = [
    "experiments/Exp1_ResNet18_Baseline_Sanitized",
    "experiments/Exp2_ResNet18_YOLO_Crop",
    "experiments/Exp3_ResNet18_CoordinateAttention"
]

results = []
for ed in exp_dirs:
    summary_file = os.path.join(ed, "test_results", "test_summary.json")
    if os.path.exists(summary_file):
        with open(summary_file, "r") as f:
            data = json.load(f)
            data["experiment"] = os.path.basename(ed)
            results.append(data)
    else:
        # Fallback: tentar ler os logs mais recentes
        log_files = glob.glob(os.path.join(ed, "*.log"))
        results.append({
            "experiment": os.path.basename(ed),
            "status": "Executado (consulte os logs em visualization/)"
        })

df = pd.DataFrame(results)
print("=== TABELA RESUMO DOS EXPERIMENTOS ===")
display(df) if "display" in globals() else print(df)


### 11. 💾 Sincronização de Resultados e Checkpoints
Salva os pesos pesados no Google Drive via DVC e comita as métricas leves (relatórios e matrizes de confusão) no repositório Git.


In [ ]:
# 1. Versionar modelos treinados e exports no DVC
!uv run dvc add experiments/*/models
!uv run dvc add experiments/*/export
!uv run dvc push

# 2. Comitar relatórios leves no Git
!git config --global user.name "BioEcoInt Lab Worker"
!git config --global user.email "sergioaugusto.bezerra.jr@gmail.com"
!git add experiments/*/*.dvc experiments/*/test_results/ experiments/*/visualization/
!git commit -m "feat(colab): treinamento dos 3 experimentos concluido com sucesso" || true
!git push origin main || true


### 12. 🛑 Desconexão Automática do Runtime (Proteção de CUs)
Para evitar que o notebook continue consumindo Compute Units (CUs) do seu plano Google AI Pro após o término dos treinamentos, a célula abaixo encerra a sessão da VM automaticamente.


In [ ]:
from google.colab import runtime

AUTO_DISCONNECT = True  # Altere para False se quiser inspecionar os arquivos antes de fechar

if AUTO_DISCONNECT:
    print("🛑 Todos os treinamentos foram concluídos! Desconectando o runtime para economizar CUs...")
    runtime.unassign()
else:
    print("⚠️ Desconexão automática desabilitada. O runtime continua ativo consumindo CUs.")
